In [2]:
import numpy as np
from qiskit import QuantumCircuit 
from qiskit.quantum_info import Statevector, Operator


# Coefficients of B
a = 1/3
b= 2/3

# prepare selction qubit
theta = 2 * np.arccos(np.sqrt(a))
print("theta in degres:", np.degrees(theta))

# qubit 0 : selection qubit
# qubit 1 : system qubit

qc = QuantumCircuit(2)

#preparing the selction state
qc.ry(theta, 0)

# apply x to the system if the selection qubit is in state |0> Apply Z to the system if the selection qubit is in state |1>
qc.x(0)
qc.cx(0, 1)
qc.x(0)  # revert the selection qubit back to its original state
qc.cz(0, 1)  # apply Z to the system if the selection qubit is in state |1>

# applt the inverse preparation
qc.ry(-theta, 0)  # apply the inverse of the selection state preparation

print(qc.draw())


### verify the final state

intial_state = Statevector.from_label('00')
# simulate the circuit
final_state = intial_state.evolve(qc)
print("Final two qubit state:")
print(final_state)

amplitudes = final_state.data
branch = np.array([amplitudes[0], amplitudes[2]])  # amplitudes of the branch where the selection qubit is in state |0>

probabilty = np.sum(np.abs(branch)**2)
conditional_state = branch / np.sqrt(probabilty)  # normalize the conditional state

print("/nProbability of the selection qubit being in state |0>:", probabilty)
print("Conditional state of the system qubit given the selection qubit is in state |0>:", conditional_state)

# Classical comparison
B = (1/3) * np.array([[0, 1], [1, 0]]) + \
    (2/3) * np.array([[1, 0], [0, -1]])

psi = np.array([1, 0], dtype=complex)
classical_result = B @ psi
classical_normalized = classical_result / np.linalg.norm(classical_result)

print("\nClassical B|0>:")
print(classical_result)

print("\nNormalized classical result:")
print(classical_normalized)

theta in degres: 109.47122063449069
     ┌────────────┐┌───┐     ┌───┐   ┌─────────────┐
q_0: ┤ Ry(1.9106) ├┤ X ├──■──┤ X ├─■─┤ Ry(-1.9106) ├
     └────────────┘└───┘┌─┴─┐└───┘ │ └─────────────┘
q_1: ───────────────────┤ X ├──────■────────────────
                        └───┘                       
Final two qubit state:
Statevector([ 0.66666667+0.j,  0.47140452+0.j,  0.33333333+0.j,
             -0.47140452+0.j],
            dims=(2, 2))
/nProbability of the selection qubit being in state |0>: 0.5555555555555556
Conditional state of the system qubit given the selection qubit is in state |0>: [0.89442719+0.j 0.4472136 +0.j]

Classical B|0>:
[0.66666667+0.j 0.33333333+0.j]

Normalized classical result:
[0.89442719+0.j 0.4472136 +0.j]
